In [ ]:
!pip install -q mediapipe opencv-python-headless

In [ ]:
!git clone https://github.com/its-bhavya/workout_narration.git
!pip install mediapipe opencv-python-headless numpy matplotlib
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
for root, dirs, files in os.walk("data"):
    depth = root.count(os.sep)
    if depth <= 2:
        print(root, "->", files[:5], f"({len(files)} files)")

In [ ]:
import pandas as pd, json, os
m = "/content/workout_narration/data/manifests/"

filtered = pd.read_csv(m + "repcount_filtered.csv")
print(filtered.columns.tolist())
print(filtered.head())

splits = pd.read_csv(m + "splits.csv")
print(splits.columns.tolist())
print(splits.head())

with open(m + "repcount_rep_boundaries.json") as f:
    bounds = json.load(f)
k = list(bounds.keys())[0]
print(k, bounds[k])

In [ ]:
import pandas as pd
import os
df = pd.read_csv("/content/workout_narration/data/manifests/repcount_filtered.csv")
print(df.columns.tolist())
df.head()

In [ ]:
squats = df[df["exercise"] == "squat"]
print(squats.head())
video_name = squats.iloc[0]["filename"]
print(video_name)

In [ ]:
import os
VIDEO_PATH = None
for root, dirs, files in os.walk("/content"):
    if video_name in files:
        VIDEO_PATH = os.path.join(root, video_name)
        break
print(VIDEO_PATH)

In [ ]:
!find /content -name "*.mp4" | head -20

In [ ]:
import json
with open("/content/workout_narration/data/manifests/repcount_rep_boundaries.json") as f:
    bounds = json.load(f)

key = "stu2_25.mp4"           # using an existing key from the JSON to verify
print(list(bounds.keys())[:5])   # check how keys are written
print(bounds[key])
print("true reps:", len(bounds[key]))

In [ ]:
print(VIDEO_PATH)

In [ ]:
import glob
VIDEO_PATH = glob.glob("/content/drive/MyDrive/synth_test_0001.mp4")[0]
print(VIDEO_PATH)

In [ ]:
# to check whats there in the video
import cv2, matplotlib.pyplot as plt
cap = cv2.VideoCapture(VIDEO_PATH)
print("fps:", cap.get(cv2.CAP_PROP_FPS),
      "frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
      "size:", int(cap.get(3)), "x", int(cap.get(4)))
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
ok, frame = cap.read()
cap.release()
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

In [ ]:
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task

In [ ]:
import cv2, numpy as np, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

options = vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="pose_landmarker.task"),
    running_mode=vision.RunningMode.VIDEO)
landmarker = vision.PoseLandmarker.create_from_options(options)

def extract_knee_angles(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    angles, i = [], 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        img = mp.Image(image_format=mp.ImageFormat.SRGB,
                       data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        res = landmarker.detect_for_video(img, int(i * 1000 / fps))
        if res.pose_landmarks:
            lm = res.pose_landmarks[0]
            angles.append(angle([lm[23].x, lm[23].y],
                                [lm[25].x, lm[25].y],
                                [lm[27].x, lm[27].y]))
        else:
            angles.append(np.nan)
        i += 1
    cap.release()
    return np.array(angles), fps

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# Helper function to compute angle between 3 points
def angle(a, b, c):
    a = np.array(a)
    b = np.array(b) # Vertex
    c = np.array(c)

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle_deg = np.abs(radians*180.0/np.pi)

    if angle_deg > 180.0:
        angle_deg = 360 - angle_deg

    return angle_deg

# 1. Extract knee angles using the previously defined function
print("Processing video... This may take a minute.")
angles, fps = extract_knee_angles(VIDEO_PATH)

# 2. Create the time array 't'
t = np.arange(len(angles)) / fps

# 3. Clean the angles and find peaks
clean = pd.Series(angles).interpolate().bfill().ffill().values
dips, _ = find_peaks(-clean, prominence=30)
print("dips found:", len(dips))

# 4. Plot the results
plt.figure(figsize=(10, 3))
plt.plot(t, clean, label="Knee Angle")
plt.plot(t[dips], clean[dips], "rx", label="Detected Reps")
plt.xlabel("time (s)")
plt.ylabel("knee angle (deg)")
plt.legend()
plt.show()